# Chapter 12 — The stellar halo in Gaia data: the Gaia-Sausage/Enceladus merger

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/galactic-dynamics-tutorial/blob/main/notebooks/12_stellar_halo.ipynb)

**Goal.** Find the debris of the last major merger of the Milky Way in the kinematics of metal-poor stars. In 2018
Gaia showed that most of the stellar halo near the Sun moves on very radial orbits: the "Gaia Sausage" (Belokurov et
al. 2018), the remains of a dwarf galaxy, "Gaia-Enceladus" (Helmi et al. 2018), that merged with the Milky Way
8–11 Gyr ago. We reproduce the main observations with Gaia DR3 and AGAMA.

**You will learn**
1. to select halo stars by chemistry, and to test the selection for contamination with kinematics;
2. to measure the **velocity anisotropy** $\beta$ (Chapter 7) and see how it changes with metallicity;
3. to recognise a **globular cluster** in phase space, and confirm it against a catalogue;
4. to map stars into **integrals of motion** — energy, angular momentum, actions (Chapters 5 and 9) — and estimate
   orbital eccentricities with AGAMA, checked against orbit integration.

**Data.** The file `halo_gse` of the tutorial's Zenodo record (https://doi.org/10.5281/zenodo.23627795): 285 683 Gaia DR3
stars with line-of-sight velocities, parallax over error above 5, and metallicity $[{\rm M/H}] < -1$ from the Gaia XP
spectra (Andrae, Rix & Chandra 2023). The selection uses chemistry only — no kinematic cut — so that kinematics can
be studied without bias. We write $[{\rm Fe/H}]$ for their $[{\rm M/H}]$; the two agree to about 0.02 dex (see the README).

**Units.** kpc, km/s, $M_\odot$; energies in (km/s)$^2$, actions and angular momenta in kpc km/s.

In [ ]:
import sys, importlib.util
if importlib.util.find_spec('agama') is None:      # install AGAMA on Colab (see Chapter 0)
    v = f'cp{sys.version_info.major}{sys.version_info.minor}'
    !pip install -q "https://github.com/vasilybelokurov/galactic-dynamics-tutorial/releases/download/agama-wheels/agama-1.0-{v}-{v}-linux_x86_64.whl"
import agama

agama.setUnits(mass=1, length=1, velocity=1)       # Msun, kpc, km/s -- before creating any AGAMA object

import os, hashlib, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Zenodo record 10.5281/zenodo.23627795 (version 1.0); as in Chapter 11
ZENODO = 'https://zenodo.org/records/23627795/files/{}?download=1'
MD5 = {'solar_neighbourhood': 'f6119c98ae50ded14fab91d1d4c071fd', 'halo_gse': '1c0533ce89d6d95ff67406dcf2a95f31',
       'disc_giants': '12f6a9ca0ac3d8ecdcd71988267b8ea7', 'rotation_curves': 'a41c95e4ecc31b3ab3bde99880736c6e',
       'dsph_gaia': '0b135e7f08816209b379f892196307b2'}


def load_data(name, cache=Path('data')):
    """Download data/<name>.parquet from Zenodo once, check its MD5, and return it as a pandas DataFrame."""
    path = cache / f'{name}.parquet'
    if not path.exists():
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} from Zenodo ...')
        urllib.request.urlretrieve(ZENODO.format(path.name), path)
    md5 = hashlib.md5(path.read_bytes()).hexdigest()
    assert md5 == MD5[name], f'{path} is corrupt (md5 {md5}); delete it and rerun'
    return pd.read_parquet(path)


# Solar parameters and the coordinate transformation of Chapter 11 §11.1
R0, z_sun = 8.122, 0.0208                  # kpc
v_sun = (12.9, 245.6, 7.78)                 # km/s
K = 4.740470463533348                       # km/s per (mas/yr * kpc)


def galactocentric(ra, dec, parallax, pmra, pmdec, vlos):
    """Gaia observables (deg, deg, mas, mas/yr, mas/yr, km/s) -> Galactocentric x, y, z [kpc] and vx, vy, vz [km/s]."""
    l, b, pml, pmb = agama.transformCelestialCoords(agama.fromICRStoGalactic, np.radians(ra), np.radians(dec), pmra, pmdec)
    return agama.getGalactocentricFromGalactic(l, b, 1 / parallax, pml * K, pmb * K, vlos,
                                               galcen_distance=R0, z_sun=z_sun, galcen_v_sun=v_sun)

## 12.1 A chemically selected halo sample, and its contaminants

The halo is a spheroid, so its natural coordinates are **spherical**: the Galactocentric distance $r$, the polar angle
$\theta$ (from the North Galactic Pole) and the azimuth $\phi$. The velocity components are

$$ v_r = \frac{\mathbf x \cdot \mathbf v}{r}, \qquad v_\theta = \frac{z v_R - R v_z}{r}, \qquad v_\phi = -\frac{L_z}{R}, $$

where $R$ and $v_R$ are the cylindrical radius and radial velocity, and $v_\phi > 0$ means rotation with the disk
(the sign convention of Chapter 11).

In [ ]:
h = load_data('halo_gse')
cols = ['ra', 'dec', 'parallax', 'pmra', 'pmdec', 'radial_velocity']
x, y, z, vx, vy, vz = galactocentric(*[h[c].to_numpy(dtype=float) for c in cols])
xv = np.column_stack([x, y, z, vx, vy, vz])        # (N, 6): the input format of AGAMA

r = np.sqrt(x**2 + y**2 + z**2)
R = np.hypot(x, y)
vR = (x * vx + y * vy) / R
vr = (x * vx + y * vy + z * vz) / r
vtheta = (z * vR - R * vz) / r
Lz = x * vy - y * vx                               # negative for disk stars in this frame (Chapter 11)
vphi = -Lz / R
d = 1 / h.parallax.to_numpy(float)
feh = h.feh.to_numpy(float)
print(f'{len(h):,} stars; heliocentric distance: median {np.median(d):.1f} kpc, 90% below {np.percentile(d, 90):.1f} kpc')
print(f'[Fe/H]: 5, 50, 95 percentiles {np.round(np.percentile(feh, [5, 50, 95]), 2)}')

A halo star does not share the disk's rotation. Yet when we look at the most metal-poor stars, $[{\rm Fe/H}] < -1.6$, within
4 kpc of the Sun, many of them move exactly like thin-disk stars: $v_\phi$ near 230 km/s and $|v_z|$ small. We count
such "cold disk" stars ($|v_\phi - 233| < 40$ and $|v_z| < 40$ km/s) as a function of height above the plane and of
apparent magnitude.

In [ ]:
cold = (np.abs(vphi - 233) < 40) & (np.abs(vz) < 40)
mp = (feh < -1.6) & (d < 4)
z_edges = [0, 0.2, 0.5, 1.0, 3.0]
frac_z = [cold[mp & (np.abs(z) >= a) & (np.abs(z) < b)].mean() for a, b in zip(z_edges[:-1], z_edges[1:])]
G = h.phot_g_mean_mag.to_numpy(float)
G_edges = [6, 12, 14, 18]
frac_G = [cold[mp & (G >= a) & (G < b)].mean() for a, b in zip(G_edges[:-1], G_edges[1:])]

fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
bins = np.linspace(-300, 400, 141)
ax[0].hist(vphi[mp & (np.abs(z) < 0.2)], bins, histtype='step', lw=2, density=True, label='$|z| < 0.2$ kpc')
ax[0].hist(vphi[mp & (np.abs(z) > 0.5)], bins, histtype='step', lw=2, density=True, label='$|z| > 0.5$ kpc')
ax[0].set_xlabel('$v_\\phi$ [km/s]'); ax[0].set_ylabel('density'); ax[0].legend(); ax[0].set_title('[Fe/H] < -1.6, d < 4 kpc')
ax[1].bar(range(4), frac_z, tick_label=[f'{a}-{b}' for a, b in zip(z_edges[:-1], z_edges[1:])], color='0.5')
ax[1].set_xlabel('$|z|$ [kpc]'); ax[1].set_ylabel('fraction of cold disk stars')
ax[2].bar(range(3), frac_G, tick_label=[f'{a}-{b}' for a, b in zip(G_edges[:-1], G_edges[1:])], color='0.5')
ax[2].set_xlabel('$G$ [mag]'); ax[2].set_ylabel('fraction of cold disk stars')
plt.tight_layout(); plt.show()
print('cold fraction by |z|:', np.round(frac_z, 3), ' by G:', np.round(frac_G, 3))
assert frac_z[0] > 0.3 and frac_z[2] < 0.05

**Figure 12.1.** Stars with $[{\rm Fe/H}] < -1.6$ within 4 kpc. Left: distribution of $v_\phi$ near the plane ($|z| < 0.2$ kpc)
and away from it ($|z| > 0.5$ kpc). Middle and right: fraction of stars with cold, thin-disk kinematics as a function
of height $|z|$ and of magnitude $G$.

Near the plane, 60% of the "very metal-poor" stars have thin-disk kinematics (Figure 12.1, left and middle); above 0.5 kpc,
only 2%. They are also mostly bright ($G < 12$). Very metal-poor stars with disk orbits exist, but not in such
numbers. The likely explanation is that the metallicities of these bright, low-latitude stars, seen through the dust
of the disk, are underestimated, so that metal-rich disk stars leak into the sample. We have not proven this cause;
the lesson is general: **a sample selected on one quantity must be checked with another.** From now on we keep stars
with $|z| > 0.5$ kpc and within 4 kpc of the Sun (parallaxes better than 20% and a well-sampled volume).

In [ ]:
clean = (np.abs(z) > 0.5) & (d < 4)
print(f'clean sample: {clean.sum():,} stars')

## 12.2 The Gaia Sausage

Figure 12.2 shows the clean sample in the $(v_r, v_\phi)$ plane in three metallicity bins.

In [ ]:
feh_bins = [(-1.3, -1.0), (-1.6, -1.3), (-3.0, -1.6)]
fig, ax = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)
for a, (lo, hi) in zip(ax, feh_bins):
    s = clean & (feh > lo) & (feh < hi)
    a.hist2d(vr[s], vphi[s], bins=120, range=[[-400, 400], [-300, 350]], norm=LogNorm(), cmap='Greys')
    a.set_title(f'{lo} < [Fe/H] < {hi}: {s.sum():,} stars')
    a.set_xlabel('$v_r$ [km/s]')
ax[0].set_ylabel('$v_\\phi$ [km/s]')
plt.tight_layout(); plt.show()

**Figure 12.2.** Spherical radial velocity $v_r$ against rotation velocity $v_\phi$ for the clean sample ($|z| > 0.5$ kpc,
$d < 4$ kpc) in three metallicity bins; log-scaled counts.

Two populations are visible in Figure 12.2. A **rotating** one, centred near $v_\phi \approx 150$ km/s with small $v_r$: the
metal-weak tail of the thick disk and the "Splash" (Belokurov et al. 2020), most prominent in the most metal-rich bin.
And a **non-rotating** one, with $v_\phi \approx 0$ but $v_r$ spreading to $\pm 300$ km/s: the Sausage. Its two lobes at
$v_r \approx \pm 200$ km/s are stars of the same debris near the Sun moving out of and into the inner Galaxy on nearly
radial orbits; few are at rest in $v_r$, because the Sun sits near the middle of their long radial excursions, where
they move fastest (compare the radial velocity of the orbits in Chapter 3). The Sausage is clearest at
$-1.6 < [{\rm Fe/H}] < -1.3$ (middle); at the lowest metallicities the distribution is rounder.

The **anisotropy parameter** (Chapter 7)

$$ \beta = 1 - \frac{\sigma_\theta^2 + \sigma_\phi^2}{2\sigma_r^2} $$

quantifies this: $\beta = 0$ for isotropic orbits, $\beta \to 1$ for purely radial ones. We compute it in narrower
metallicity bins, together with the mean rotation. The dispersions are plain variances (`np.var` subtracts the mean,
so rotation does not count as dispersion), and the uncertainty is from bootstrap resampling of the stars.

In [ ]:
def beta(s):
    return 1 - (np.var(vtheta[s]) + np.var(vphi[s])) / (2 * np.var(vr[s]))


edges = np.array([-3.5, -2.4, -2.0, -1.8, -1.6, -1.4, -1.2, -1.0])
centres, B, B_err, V = [], [], [], []
rng = np.random.default_rng(0)
for lo, hi in zip(edges[:-1], edges[1:]):
    idx = np.flatnonzero(clean & (feh > lo) & (feh < hi))
    boot = [beta(rng.choice(idx, len(idx))) for _ in range(100)]    # resample stars with replacement
    centres.append(np.median(feh[idx])); B.append(beta(idx)); B_err.append(np.std(boot)); V.append(vphi[idx].mean())
    print(f'{lo:5.1f} < [Fe/H] < {hi:5.1f}: N = {len(idx):6,}, beta = {B[-1]:.2f} +- {B_err[-1]:.2f}, mean v_phi = {V[-1]:5.1f} km/s')

fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
ax[0].errorbar(centres, B, B_err, fmt='ko-')
ax[0].set_xlabel('[Fe/H]'); ax[0].set_ylabel('$\\beta$'); ax[0].set_ylim(0, 1)
ax[1].plot(centres, V, 'ko-')
ax[1].set_xlabel('[Fe/H]'); ax[1].set_ylabel('mean $v_\\phi$ [km/s]')
plt.tight_layout(); plt.show()

assert max(B) - B[0] > 0.2 and np.argmax(B) in (4, 5)

**Figure 12.3.** Left: velocity anisotropy $\beta$ of the clean sample in metallicity bins (bootstrap errors). Right: the mean
rotation velocity in the same bins.

The anisotropy rises from about 0.35 for the most metal-poor stars to about 0.64 at $-1.8 < [{\rm Fe/H}] < -1.2$ (Figure 12.3, left),
and drops again towards $-1$, where the rotating disk stars become numerous (right). Belokurov et al. (2018) found the
same trend with $\beta$ reaching about 0.9 for $[{\rm Fe/H}] > -1.7$; they modelled the halo separately from the disk,
whereas our single $\beta$ per bin mixes the radial Sausage with the rotating stars, which lowers it. Their
interpretation: the radially anisotropic stars are the debris of one massive dwarf galaxy, whose orbit was radialised
before it was disrupted — the last major merger of the Milky Way.

## 12.3 A globular cluster in phase space

In the most metal-poor panel of Figure 12.2 there is a compact dark spot near $(v_r, v_\phi) \approx (50, 100)$ km/s. A
single dense knot in velocity space suggests a group of stars moving together: a star cluster. Where are these stars on
the sky?

In [ ]:
knot = clean & (feh < -1.6) & (np.abs(vr - 50) < 15) & (np.abs(vphi - 100) < 15)
ra, dec = h.ra.to_numpy(float), h.dec.to_numpy(float)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
# the densest spot on the sky: the knot star with the most knot neighbours within 1 degree
sep = lambda a, b: np.hypot((ra[a] - ra[b]) * np.cos(np.radians(dec[b])), dec[a] - dec[b])   # small-angle separation [deg]
ik = np.flatnonzero(knot)
centre = ik[np.argmax([(sep(ik, i) < 1).sum() for i in ik])]
core = knot & (sep(np.arange(len(h)), centre) < 1)           # knot stars within 1 degree of that spot
ax[0].plot(ra[knot & ~core], dec[knot & ~core], '.', color='0.6', ms=3)
ax[0].plot(ra[core], dec[core], 'r.', ms=3)
ax[0].plot(ra[centre], dec[centre], 'o', ms=16, mfc='none', mec='r', mew=1.5)
ax[0].set_xlabel('RA [deg]'); ax[0].set_ylabel('Dec [deg]'); ax[0].set_xlim(360, 0); ax[0].set_title(f'{knot.sum()} stars of the knot')
ax[1].plot(h.pmra[knot & ~core], h.pmdec[knot & ~core], '.', color='0.6', ms=3, label='knot, elsewhere on the sky')
ax[1].plot(h.pmra[core], h.pmdec[core], 'r.', ms=3, label='knot, within 1° of the densest spot')
ax[1].set_xlabel('$\\mu_{\\alpha*}$ [mas/yr]'); ax[1].set_ylabel('$\\mu_\\delta$ [mas/yr]'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

# NGC 6397 in Vasiliev & Baumgardt (2021, table A1, VizieR J/MNRAS/505/5978): position, proper motion, parallax
ngc6397 = dict(ra=265.175, dec=-53.674, pmra=3.260, pmdec=-17.664, parallax=0.416)
for k, ref in ngc6397.items():
    print(f'{k:8s}: median of the core stars {h[k][core].median():8.3f}   catalogue {ref:8.3f}')
print(f'{core.sum()} of the {knot.sum()} knot stars lie within 1 degree of its centre')
assert abs(h.pmra[core].median() - 3.260) < 0.3 and abs(h.pmdec[core].median() + 17.664) < 0.3
assert abs(h.ra[core].median() - 265.175) < 0.3 and abs(h.dec[core].median() + 53.674) < 0.3

**Figure 12.4.** Stars in the velocity-space knot ($[{\rm Fe/H}] < -1.6$, $v_r$ and $v_\phi$ within 15 km/s of (50, 100) km/s).
Left: positions on the sky. Right: proper motions. Red: stars within 1° of the densest spot on the sky (circled).

About half of the knot stars sit in one spot of the sky (Figure 12.4, left, red), and these share one proper motion
(right); the rest are field halo stars that happen to have similar velocities. The proper motion and parallax match the nearby globular cluster **NGC 6397**, about 2.4 kpc away (parallax 0.42 mas). Our median parallax is 0.04 mas smaller: Gaia DR3 parallaxes have a known
zero-point offset of a few hundredths of a mas, which the catalogue corrects (Lindegren et al. 2021) and our file does not. A cluster is bound, so its stars stay together in all six phase-space coordinates; dissolved
debris like the Sausage stays together only in the integrals of motion, which is the subject of the next section. The
cluster is a small part of the sample and does not affect the conclusions; we leave it in.

## 12.4 Integrals of motion

Debris from a disrupted satellite spreads along its orbit and phase-mixes (Chapter 6) until it fills a large volume of
space. But its stars keep their **integrals of motion**, approximately those of the satellite: energy $E$, angular momentum
$L_z$ and the actions. We compute them in McMillan's (2017) potential, as in Chapter 11. AGAMA gives the potential at
many points at once with `pot.potential(points)`, and `pot.Rperiapo(xv)` estimates the pericentre and apocentre of each
orbit, from which the eccentricity $e = (r_{\rm apo} - r_{\rm peri}) / (r_{\rm apo} + r_{\rm peri})$.

`Rperiapo` with an $(N, 6)$ array treats the potential as if it were spherical (it uses the total angular momentum), so
for orbits that feel the flattened disk it is an approximation. We check it against the minimum and maximum radius of
300 orbits integrated for 3 time units (2.9 Gyr).

In [ ]:
pot = agama.Potential(os.path.join(os.path.dirname(agama.__file__), 'data', 'McMillan17.ini'))
E = pot.potential(xv[:, :3]) + 0.5 * (vx**2 + vy**2 + vz**2)        # energy per unit mass [(km/s)^2]
peri_apo = pot.Rperiapo(xv)                                        # (N, 2) array
ecc = (peri_apo[:, 1] - peri_apo[:, 0]) / (peri_apo[:, 1] + peri_apo[:, 0])

test = np.random.default_rng(1).choice(np.flatnonzero(clean), 300, replace=False)
orbits = agama.orbit(potential=pot, ic=xv[test], time=3.0, trajsize=3000)
r_orb = [np.linalg.norm(traj[:, :3], axis=1) for times, traj in orbits]   # radius along each orbit
ecc_orb = np.array([(q.max() - q.min()) / (q.max() + q.min()) for q in r_orb])
diff = np.abs(ecc_orb - ecc[test])
print(f'eccentricity: Rperiapo vs orbit integration: median |difference| {np.median(diff):.3f}, 90th percentile {np.percentile(diff, 90):.3f}')
assert np.median(diff) < 0.05

The estimate agrees with the integrated orbits to 0.04 for a typical star, and to about 0.15 for 90% of them; good
enough to separate radial from circular orbits. (Integrating all orbits would be more accurate, and takes a few minutes.)

Now the energy–angular momentum plane (Figure 12.5) and the "action diamond" (Figure 12.6). For the diamond, AGAMA's
`ActionFinder` gives $(J_r, J_z, J_\phi)$; with $J_{\rm tot} = J_r + J_z + |J_\phi|$, the horizontal axis $J_\phi/J_{\rm tot}$ runs
from retrograde ($-1$) to prograde circular orbits (+1), and the vertical axis $(J_z - J_r)/J_{\rm tot}$ from radial orbits
($-1$, bottom corner) to polar ones (+1, top). We flip the sign of $L_z$ and $J_\phi$ so that prograde (disk-like) is positive.

In [ ]:
J = agama.ActionFinder(pot)(xv)
Jr, Jz, Jphi = J.T
Jtot = Jr + Jz + np.abs(Jphi)
radial = ecc > 0.8                                    # our working definition of "Sausage-like" orbits
print(f'stars with NaN actions (unbound or failed): {np.isnan(J).any(axis=1).sum()}')
print(f'clean sample: {np.mean(radial[clean]):.2f} have e > 0.8')

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
s = clean
ax[0].hist2d(-Lz[s], E[s] / 1e5, bins=200, range=[[-3000, 4000], [-2.1, -0.6]], norm=LogNorm(), cmap='Greys')
ax[0].plot(-Lz[s & radial], E[s & radial] / 1e5, ',', color='C3', alpha=0.3)
ax[0].set_xlabel('$-L_z$ [kpc km/s]'); ax[0].set_ylabel('$E$ [$10^5$ (km/s)$^2$]'); ax[0].set_title('red: $e > 0.8$')
H = ax[1].hist2d(-Jphi[s] / Jtot[s], (Jz[s] - Jr[s]) / Jtot[s], bins=120, range=[[-1, 1], [-1, 1]], norm=LogNorm(), cmap='Greys')
ax[1].plot(-Jphi[s & radial] / Jtot[s & radial], (Jz[s & radial] - Jr[s & radial]) / Jtot[s & radial], ',', color='C3', alpha=0.3)
ax[1].set_xlabel('$-J_\\phi / J_{\\rm tot}$'); ax[1].set_ylabel('$(J_z - J_r) / J_{\\rm tot}$')
plt.tight_layout(); plt.show()

**Figure 12.5.** Left: energy against angular momentum (sign flipped so that disk rotation is positive) for the clean
sample; log-scaled counts, with stars on orbits of eccentricity $e > 0.8$ overplotted in red. Right: the action diamond,
$J_\phi/J_{\rm tot}$ against $(J_z - J_r)/J_{\rm tot}$, with the same $e > 0.8$ stars in red.

In Figure 12.5 (left), the disk stars form the dense sequence running to the right along the lower envelope (the
circular orbits have the lowest energy at each $L_z$). The radial orbits ($e > 0.8$, red) form a narrow vertical
column at $L_z \approx 0$ over a wide range of energies: the Sausage. In the action diamond (right), the disk crowds into the right-hand corner (prograde, nearly circular),
and the Sausage into the bottom corner: orbits dominated by $J_r$.

## 12.5 The metallicity of the radial debris

If the radial orbits are the debris of one galaxy, their stars should share that galaxy's chemistry. Figure 12.6 shows,
for each metallicity bin, the fraction of stars on orbits with $e > 0.8$, and the metallicity distributions of stars on
radial and on low-eccentricity orbits.

In [ ]:
frac_radial = [radial[clean & (feh > lo) & (feh < hi)].mean() for lo, hi in zip(edges[:-1], edges[1:])]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
ax[0].plot(centres, frac_radial, 'ko-')
ax[0].set_xlabel('[Fe/H]'); ax[0].set_ylabel('fraction with $e > 0.8$')
fb = np.linspace(-3, -1, 41)
ax[1].hist(feh[clean & radial], fb, histtype='step', lw=2, density=True, label='$e > 0.8$')
ax[1].hist(feh[clean & (ecc < 0.5)], fb, histtype='step', lw=2, density=True, label='$e < 0.5$')
ax[1].set_xlabel('[Fe/H]'); ax[1].set_ylabel('density'); ax[1].legend()
plt.tight_layout(); plt.show()
print('fraction with e > 0.8 per bin:', np.round(frac_radial, 2))
assert np.argmax(frac_radial) in (3, 4, 5) and max(frac_radial) - frac_radial[0] > 0.05

**Figure 12.6.** Left: fraction of the clean sample on orbits with $e > 0.8$, in the metallicity bins of Figure 12.3.
Right: normalised metallicity distributions of stars with $e > 0.8$ and with $e < 0.5$.

The radial orbits are most common at $-1.8 \lesssim [{\rm Fe/H}] \lesssim -1.2$ (Figure 12.6, left) and rarer among the most metal-poor
stars. Our sample stops at $[{\rm Fe/H}] = -1$, so we see only the metal-poor side of the debris; spectroscopic surveys
that reach higher metallicities find Sausage stars above $-1$ as well. A galaxy that enriched its gas to these levels was
massive for a dwarf: Belokurov et al. (2018) estimate a virial mass above $10^{10}\,M_\odot$ for the progenitor. The
low-eccentricity stars (right) are the
disk-like population of Figure 12.2 and are more metal-rich.

## Summary

- Chemical selections need kinematic checks: 60% of the "very metal-poor" stars near the plane have thin-disk kinematics; a
  cut $|z| > 0.5$ kpc removes them.
- The halo near the Sun is radially anisotropic, most strongly at $-1.8 \lesssim [{\rm Fe/H}] \lesssim -1.2$: the Gaia Sausage,
  the debris of the Gaia-Sausage/Enceladus merger.
- A bound cluster (NGC 6397) is compact in all of phase space; dissolved debris is compact only in the integrals of motion.
- AGAMA gives energies, actions and approximate peri/apocentres for $10^5$ stars in under a second; check approximations
  against orbit integration.

| task | AGAMA |
|---|---|
| potential at many points | `pot.potential(points)` with `points` of shape $(N, 3)$ |
| peri- and apocentre | `pot.Rperiapo(xv)` (spherical approximation for $(N, 6)$ input) or `pot.Rperiapo(E, L)` |
| actions | `agama.ActionFinder(pot)(xv)` → $(J_r, J_z, J_\phi)$ |
| orbit check | `agama.orbit(potential=pot, ic=xv, time=T, trajsize=n)` |

## Exercises

1. **Mixture model.** Fit the $(v_r, v_\theta, v_\phi)$ distribution of each metallicity bin with a sum of Gaussians
   (`sklearn.mixture.GaussianMixture`). How many components do you need to describe the two lobes of the Sausage?
   Why does a single Gaussian with large $\sigma_r$ fail?
2. **The potential.** Recompute $E$ and the eccentricities in the halo-fit potential of Chapter 11 §11.4. Which stars
   change their classification $e > 0.8$?
3. **Retrograde debris.** Select stars with $-L_z < -1000$ kpc km/s and high energy. How does their metallicity distribution
   compare with the Sausage's? (This region holds the debris called Sequoia.)
4. **Remove the cluster.** Remove NGC 6397's stars (all stars within 0.5° of its centre) and recompute $\beta$ for the most
   metal-poor bins. Does it change?

## References

- R. Andrae, H.-W. Rix & V. Chandra (2023), "Robust data-driven metallicities for 175 million stars from Gaia XP spectra",
  ApJS 267, 8: https://ui.adsabs.harvard.edu/abs/2023ApJS..267....8A
- V. Belokurov et al. (2018), "Co-formation of the disc and the stellar halo", MNRAS 478, 611:
  https://ui.adsabs.harvard.edu/abs/2018MNRAS.478..611B
- V. Belokurov et al. (2020), "The biggest splash", MNRAS 494, 3880: https://ui.adsabs.harvard.edu/abs/2020MNRAS.494.3880B
- A. Helmi et al. (2018), "The merger that led to the formation of the Milky Way's inner stellar halo and thick disk",
  Nature 563, 85: https://ui.adsabs.harvard.edu/abs/2018Natur.563...85H
- L. Lindegren et al. (2021), "Gaia Early Data Release 3. Parallax bias versus magnitude, colour, and position", A&A 649, A4:
  https://ui.adsabs.harvard.edu/abs/2021A&A...649A...4L
- P. McMillan (2017), MNRAS 465, 76: https://ui.adsabs.harvard.edu/abs/2017MNRAS.465...76M
- E. Vasiliev & H. Baumgardt (2021), "Gaia EDR3 view on galactic globular clusters", MNRAS 505, 5978:
  https://ui.adsabs.harvard.edu/abs/2021MNRAS.505.5978V
- Data: V. Belokurov (2026), "Galactic Dynamics Tutorial data", Zenodo, https://doi.org/10.5281/zenodo.23627795 — please also cite
  the sources listed in its README (Gaia DR3, Andrae et al. 2023, Katz et al. 2023).